# Parsing PeeringDB files

- This notebook parse the peeringDB json file and extract the maximum prefix limit for each ASN.
- Also, perform two preprocessing steps: 
    - fill the missing dates using the most closer value
    - increase the temporal resolution by resampling the data by 3 to match the RIBs time series

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import datetime
import urllib.request
from tqdm import tqdm
from matplotlib import pyplot as plt

## Confs

In [ ]:
import os
import json

REPO_ROOT = os.path.abspath("..")  # notebooks/ -> repo root
with open(os.path.join(REPO_ROOT, "settings.json")) as fd:
    parameters = json.load(fd)

def resolve(p):
    return p if os.path.isabs(p) else os.path.normpath(os.path.join(REPO_ROOT, p))

try:
    data_dir = resolve(parameters["DATA_DIR"])
    data_raw_dir = resolve(parameters["DATA_RAW_DIR"])
    start_date = parameters["START_DATE"]
    end_date = parameters["END_DATE"]
except Exception:
    raise ValueError("Invalid parameter file")


## Load data

### Merge files

In [ ]:
dates = []
init_date = datetime.datetime(2024, 12, 31)  # January 1, 2025
delta = datetime.timedelta(days=1)

date = init_date
while date < datetime.datetime(2026, 1, 1):
    dates.append(date)
    date += delta

In [ ]:
files = os.listdir(f"{data_raw_dir}/peeringdb")
files = sorted(files)
files = [f"{data_raw_dir}/peeringdb/{f}" for f in files if f.endswith(".json")]

In [ ]:
dfs = []

for file in tqdm(files):
    filename = os.path.basename(file)

    if not os.path.exists(file):
        continue

    fd = open(file, "r")
    data = json.load(fd)
    fd.close()

    timestamp = data["net"]["meta"]["generated"]
    date = datetime.datetime.fromtimestamp(timestamp).date()

    if date <= datetime.datetime(2025, 1, 1).date():
        continue

    df = pd.DataFrame(data["net"]["data"])
    df["date"] = date

    df["updated"] = df["updated"].apply(
        lambda updated: datetime.datetime.strptime(updated, "%Y-%m-%dT%H:%M:%SZ")
    )
    df = df[
        [
            "asn",
            "name",
            "aka",
            "info_type",
            "updated",
            "date",
            "info_prefixes6",
            "info_prefixes4",
        ]
    ].copy()
    df = df.rename(
        columns={
            "info_prefixes4": "limit_ipv4",
            "info_prefixes6": "limit_ipv6",
        }
    )

    df.drop_duplicates(subset=["asn", "date"], inplace=True)

    df.dropna(inplace=True)

    df["asn"] = df["asn"].astype(int)
    df["limit_ipv4"] = df["limit_ipv4"].astype(int)
    df["limit_ipv6"] = df["limit_ipv6"].astype(int)

    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

df_all = (
    df_all.groupby(["asn", "name", "aka", "info_type"])
    .agg(
        {
            "date": lambda x: list(x),
            "limit_ipv4": lambda x: list(x),
            "limit_ipv6": lambda x: list(x),
            "updated": lambda x: max(x) if (x != pd.Timestamp(0)).any() else None,
        }
    )
    .reset_index()
)

df_all.rename(
    columns={"limit_ipv4": "limits_ipv4", "limit_ipv6": "limits_ipv6", "date": "dates"},
    inplace=True,
)


df_all.sort_values(by=["asn", "updated"], inplace=True)
df_all.drop_duplicates(subset=["asn"], keep="last", inplace=True)

df_all = df_all.reset_index(drop=True)
df_all.head(2)

In [ ]:
def fill_missing_dates(row):
    asn = row.asn
    limit_dates = row["dates"]
    limits_ipv4 = row["limits_ipv4"]
    limits_ipv6 = row["limits_ipv6"]

    new_prefixes_ipv4 = []
    new_prefixes_ipv6 = []
    try:
        limit_index = 0
        for date in dates[1:]:
            new_prefixes_ipv4.append(limits_ipv4[limit_index])
            new_prefixes_ipv6.append(limits_ipv6[limit_index])

            if date.date() == limit_dates[limit_index]:
                if limit_index < len(limit_dates) - 1:
                    limit_index += 1

    except Exception as e:
        print(asn, e)

    if set(new_prefixes_ipv4) == {0}:
        new_prefixes_ipv4 = None
    else:
        new_prefixes_ipv4 = np.array(new_prefixes_ipv4)
        non_zero_min = np.min(new_prefixes_ipv4[new_prefixes_ipv4 > 0])
        new_prefixes_ipv4[new_prefixes_ipv4 == 0] = non_zero_min
        new_prefixes_ipv4 = list(new_prefixes_ipv4)

    if set(new_prefixes_ipv6) == {0}:
        new_prefixes_ipv6 = None
    else:
        new_prefixes_ipv6 = np.array(new_prefixes_ipv6)
        non_zero_min = np.min(new_prefixes_ipv6[new_prefixes_ipv6 > 0])
        new_prefixes_ipv6[new_prefixes_ipv6 == 0] = non_zero_min
        new_prefixes_ipv6 = list(new_prefixes_ipv6)

    return dates[1:], new_prefixes_ipv4, new_prefixes_ipv6

In [ ]:
df_all[["new_dates", "new_limits_ipv4", "new_limits_ipv6"]] = df_all.apply(
    fill_missing_dates, axis=1, result_type="expand"
)

df_all.rename(
    columns={
        "limits_ipv4": "raw_limits_ipv4",
        "limits_ipv6": "raw_limits_ipv6",
        "dates": "raw_dates",
    },
    inplace=True,
)
df_all.rename(
    columns={
        "new_limits_ipv4": "limits_ipv4",
        "new_limits_ipv6": "limits_ipv6",
        "new_dates": "dates",
    },
    inplace=True,
)

In [ ]:
def increase_time_resolution(row):

    limits_ipv4 = row["limits_ipv4"]
    limits_ipv6 = row["limits_ipv6"]
    limits_dates = row["dates"]

    new_limits_ipv4 = None
    new_limits_ipv6 = None
    new_limits_dates = []

    for i in range(len(limits_dates)):
        for delta in [0, 8, 16]:
            new_limits_dates.append(limits_dates[i] + datetime.timedelta(hours=delta))

    if limits_ipv4 is not None:
        new_limits_ipv4 = []
        for i in range(len(limits_ipv4)):
            for delta in [0, 8, 16]:
                new_limits_ipv4.append(limits_ipv4[i])
        new_limits_ipv4 = [int(x) for x in new_limits_ipv4]

    if limits_ipv6 is not None:
        new_limits_ipv6 = []
        for i in range(len(limits_ipv6)):
            for delta in [0, 8, 16]:
                new_limits_ipv6.append(limits_ipv6[i])
        new_limits_ipv6 = [int(x) for x in new_limits_ipv6]

    return new_limits_dates, new_limits_ipv4, new_limits_ipv6

In [ ]:
df_all[["dates", "limits_ipv4", "limits_ipv6"]] = df_all.apply(
    increase_time_resolution, axis=1, result_type="expand"
)

### Check

In [ ]:
row = df_all[df_all["asn"] == 6453].iloc[0]

plt.figure(figsize=(12, 6))

plt.plot(
    row["raw_dates"],
    row["raw_limits_ipv4"],
    lw=8,
    alpha=0.5,
    color="tab:blue",
    label="IPv4 Prefixes Limit",
)
plt.plot(
    row["raw_dates"],
    row["raw_limits_ipv6"],
    lw=8,
    alpha=0.5,
    color="tab:green",
    label="IPv6 Prefixes Limit",
)

plt.plot(row["dates"], row["limits_ipv4"], lw=1, alpha=1, color="tab:blue")
plt.plot(row["dates"], row["limits_ipv6"], lw=1, alpha=1, color="tab:green")

plt.xlabel("Date")
plt.ylabel("Number of Prefixes")
plt.title(f'ASN {row["asn"]} - {row["name"]}')
plt.xlim(row["dates"][0], row["dates"][-1])
plt.xticks(rotation=0)
plt.legend()
plt.show()

## Save data

In [ ]:
set(df_all["info_type"])

In [ ]:
def infer_info_type(row):
    name = row["name"]
    name = name.lower()

    info_type = row["info_type"]

    if info_type != "":
        return info_type

    if "university" in name or "research" in name:
        return "Educational/Research"
    elif "government" in name:
        return "Government"
    else:
        return "N/A"


df_all["info_type"] = df_all.apply(infer_info_type, axis=1)

In [ ]:
filename = (
    f"{data_dir}/processed/peeringdb/prefix_limit_peeringdb_{start_date}_{end_date}.pkl"
)
df_all.to_pickle(filename)
df_all.head(2)